In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")

    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth

!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
# Then proceed with your imports
from unsloth import FastLanguageModel
from datasets import load_dataset
from google.colab import userdata
import torch
max_seq_length = 4096
load_in_4bit = True  # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",  # Llama-3.1 2x faster
    "unsloth/Mistral-Small-Instruct-2409",  # Mistral 22b 2x faster!
    "unsloth/Phi-4",  # Phi-4 2x faster!
    "unsloth/Phi-4-unsloth-bnb-4bit",  # Phi-4 Unsloth Dynamic 4-bit Quant
    "unsloth/gemma-2-9b-bnb-4bit",  # Gemma 2x faster!
    "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",  # Qwen 2.5 2x faster!
    "unsloth/Llama-3.2-1B-bnb-4bit",  # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
]  # More models at https://unsloth.ai/docs/get-started/all-our-models

hf_token = userdata.get('HF_TOKEN')
print("Loading fine-tuned model from Hugging Face...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Jaiccc/model_0_streaming_timestamp", # Your private repo
    max_seq_length = 4096,
    load_in_4bit = True,
    token = hf_token,
)

# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

Loading fine-tuned model from Hugging Face...
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Unsloth 2026.3.4 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(100352, 5120, padding_idx=100351)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=5120, out_features=5120, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=5120, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=5120, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:
# System prompt identical to training
SYSTEM_PROMPT = """Your task is to analyze terminal XML logs and determine whether the timestamp in the TARGET LINE belongs to a "new event" or an "old event".

### DEFINITION OF A NEW EVENT:
1. **Explicit Prompts:** The very first `<user_input>` that immediately follows a shell prompt (e.g., `demo@faiserver:~$`).
2. **Phase Transitions:** In automated logs, moving from one major build stage to another (e.g., from 'fai-mirror finished' to 'Copying the nfsroot').
3. **Internal Logic:** Shifts from downloading to processing.

### WHAT IS *NOT* A NEW EVENT (OLD EVENT):
- **User Input / Keystrokes:** A user typing a command, including pressing the Enter key (a newline `\\n`), is just the completion of the input phase.
- **Incomplete Tasks:** Continuous system output without a clear phase shift.

CRITICAL INSTRUCTION: You must classify ONLY the timestamp found in the "### TARGET LINE" section. Do NOT extract timestamps from the "### CONTEXT" section. Output only the timestamp and the classification. Do NOT use brackets, periods, explanations, or markdown formatting.

Output Format Example 1: 39.229814, old event
Output Format Example 2: 111.602501, new event"""


# Regex compilers
CHUNK_RE = re.compile(r"<(user_input|system_output)\b[^>]*?(?:/>|>.*?</\1>)", flags=re.DOTALL)
TIME_RE = re.compile(r'timestamp="([\d\.]+)"')

def get_timestamp(chunk_text: str) -> str:
    match = TIME_RE.search(chunk_text)
    return match.group(1) if match else ""

def truncate_single_chunk(raw_text: str, tag_type: str, max_lines: int = 15) -> str:
    """Compresses massive system outputs to save context length (matches training logic)."""
    if tag_type != "system_output" or raw_text.endswith("/>"):
        return raw_text

    first_close = raw_text.find(">")
    last_open = raw_text.rfind("</")

    if first_close == -1 or last_open == -1: return raw_text

    opening_tag = raw_text[:first_close+1]
    closing_tag = raw_text[last_open:]
    inner_text = raw_text[first_close+1:last_open]

    lines = inner_text.split('\n')
    if len(lines) > max_lines:
        head = '\n'.join(lines[:5])
        tail = '\n'.join(lines[-5:])
        removed = len(lines) - 10
        return f"{opening_tag}{head}\n\n... [TRUNCATED {removed} LINES] ...\n\n{tail}{closing_tag}"

    return raw_text

In [ ]:
import re
from google.colab import files
from tqdm.notebook import tqdm

# ==========================================
# 1. SETUP & HELPER FUNCTIONS
# ==========================================
SYSTEM_PROMPT = """Your task is to analyze terminal XML logs and determine whether the timestamp in the TARGET LINE belongs to a "new event" or an "old event".

### DEFINITION OF A NEW EVENT:
1. **Explicit Prompts:** The very first `<user_input>` that immediately follows a shell prompt (e.g., `demo@faiserver:~$`).
2. **Phase Transitions:** In automated logs, moving from one major build stage to another (e.g., from 'fai-mirror finished' to 'Copying the nfsroot').
3. **Internal Logic:** Shifts from downloading to processing.

### WHAT IS *NOT* A NEW EVENT (OLD EVENT):
- **User Input / Keystrokes:** A user typing a command, including pressing the Enter key (a newline `\n`), is just the completion of the input phase.
- **Incomplete Tasks:** Continuous system output without a clear phase shift.

CRITICAL INSTRUCTION: You must classify ONLY the timestamp found in the "### TARGET LINE" section. Do NOT extract timestamps from the "### CONTEXT" section. Output only the timestamp and the classification. Do NOT use brackets, periods, explanations, or markdown formatting."""

CHUNK_RE = re.compile(r"<(user_input|system_output)\b[^>]*?(?:/>|>.*?</\1>)", flags=re.DOTALL)
TIME_RE = re.compile(r'timestamp="([\d\.]+)"')

def get_timestamp(chunk_text: str) -> str:
    match = TIME_RE.search(chunk_text)
    return match.group(1) if match else ""

def truncate_single_chunk(raw_text: str, tag_type: str, max_lines: int = 15) -> str:
    if tag_type != "system_output" or raw_text.endswith("/>"):
        return raw_text
    first_close = raw_text.find(">")
    last_open = raw_text.rfind("</")
    if first_close == -1 or last_open == -1: return raw_text

    opening_tag = raw_text[:first_close+1]
    closing_tag = raw_text[last_open:]
    inner_text = raw_text[first_close+1:last_open]

    lines = inner_text.split('\n')
    if len(lines) > max_lines:
        head = '\n'.join(lines[:5])
        tail = '\n'.join(lines[-5:])
        removed = len(lines) - 10
        return f"{opening_tag}{head}\n\n... [TRUNCATED {removed} LINES] ...\n\n{tail}{closing_tag}"
    return raw_text

def compress_context_window(chunks, max_total_lines=25):
    total_lines = sum(len(c["text"].split('\n')) for c in chunks)
    if total_lines <= max_total_lines:
        return "\n".join([c["text"] for c in chunks])

    result = []
    for idx, c in enumerate(chunks):
        raw_text = c["text"]
        if idx < 5 or idx >= len(chunks) - 5:
            result.append(raw_text)
        else:
            if "<system_output" in raw_text and not raw_text.endswith("/>"):
                first_close = raw_text.find(">")
                last_open = raw_text.rfind("</")
                if first_close != -1 and last_open != -1:
                    opening_tag = raw_text[:first_close+1]
                    closing_tag = raw_text[last_open:]
                    result.append(f"{opening_tag}... [TRUNCATED TO SAVE SPACE] ...{closing_tag}")
                else:
                    result.append(raw_text)
            else:
                result.append(raw_text)
    return "\n".join(result)

# ==========================================
# 2. UPLOAD & INFERENCE LOOP
# ==========================================
print("Please upload a raw XML terminal log:")
uploaded = files.upload()

# This list will store only the timestamps identified as NEW
collected_new_events = []

for filename, content in uploaded.items():
    print(f"\nProcessing {filename}...")
    xml_content = content.decode('utf-8')

    all_chunks = []
    for match in CHUNK_RE.finditer(xml_content):
        tag_type = match.group(1)
        raw_text = match.group(0)

        if len(raw_text) > 4000:
            raw_text = raw_text[:2000] + "\n...[MASSIVE LINE TRUNCATED]...\n" + raw_text[-2000:]

        processed_text = truncate_single_chunk(raw_text, tag_type, max_lines=15)
        ts_str = get_timestamp(processed_text)
        if ts_str:
            all_chunks.append({"ts": ts_str, "text": processed_text})

    if not all_chunks:
        print("No valid timestamps found.")
        continue

    print(f"Starting inference with 10-step skip logic on {len(all_chunks)} events...\n")

    # --- SKIP LOGIC INITIALIZATION ---
    skip_counter = 0

    for i in range(len(all_chunks)):
        target_chunk = all_chunks[i]
        target_ts = target_chunk["ts"]

        # Check if we are currently in a "skipping" phase
        if skip_counter > 0:
            model_result = "old event (auto-skipped)"
            skip_counter -= 1
        else:
            # Prepare Context and Prompt (Only if not skipping)
            start_idx = max(0, i - 14)
            window_chunks = all_chunks[start_idx : i + 1]
            context_chunks = window_chunks[:-1]

            if len(context_chunks) > 0:
                context_text = compress_context_window(context_chunks, max_total_lines=25)
            else:
                context_text = "No previous context available."

            target_text = target_chunk["text"]

            # Standard Truncation
            if len(context_text) > 8000: context_text = context_text[-8000:]
            if len(target_text) > 2000: target_text = target_text[:1000] + "..." + target_text[-1000:]

            combined_user_text = f"{SYSTEM_PROMPT}\n\n### CONTEXT:\n{context_text}\n\n### TARGET LINE:\n{target_text}"
            prompt = f"<|im_start|>user<|im_sep|>{combined_user_text}<|im_end|><|im_start|>assistant<|im_sep|>"

            inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

            # Emergency Slice
            if inputs.input_ids.shape[1] > 4000:
                inputs.input_ids = inputs.input_ids[:, -4000:]
                inputs.attention_mask = inputs.attention_mask[:, -4000:]

            outputs = model.generate(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=32,
                temperature=0.1,
                pad_token_id=tokenizer.eos_token_id
            )

            raw_output = tokenizer.batch_decode(outputs, clean_up_tokenization_spaces=True)[0]
            m = re.search(r'<\|im_start\|>assistant<\|im_sep\|>(.*?)<\|im_end\|>', raw_output, re.S)
            model_result = m.group(1).strip().lower() if m else "old event"

        # --- PROCESS RESULTS & MANAGE COUNTER ---
        if "new" in model_result and "auto-skipped" not in model_result:
            icon = "🔥 NEW EVENT"
            collected_new_events.append(target_ts)
            # Trigger the skip for the next 10 timestamps
            skip_counter = 10
        else:
            icon = "⚪ old event"

        print(f"[{i+1:03d}/{len(all_chunks):03d}] TS: {target_ts:<10} | {icon} | Output: '{model_result}'")

    print("\n" + "="*30)
    print("COLLECTED NEW EVENT TIMESTAMPS:")
    print(collected_new_events)
    print("="*30)

Please upload a raw XML terminal log:


Saving first_julia_recording.asciinema.xml to first_julia_recording.asciinema.xml

Processing first_julia_recording.asciinema.xml...
Starting inference with 10-step skip logic on 193 events...

[001/193] TS: 0.096022   | 🔥 NEW EVENT | Output: '0.096022, new event'
[002/193] TS: 9.163614   | ⚪ old event | Output: 'old event (auto-skipped)'
[003/193] TS: 9.164051   | ⚪ old event | Output: 'old event (auto-skipped)'
[004/193] TS: 9.365744   | ⚪ old event | Output: 'old event (auto-skipped)'
[005/193] TS: 9.366263   | ⚪ old event | Output: 'old event (auto-skipped)'
[006/193] TS: 9.589844   | ⚪ old event | Output: 'old event (auto-skipped)'
[007/193] TS: 9.59026    | ⚪ old event | Output: 'old event (auto-skipped)'
[008/193] TS: 9.708352   | ⚪ old event | Output: 'old event (auto-skipped)'
[009/193] TS: 9.708844   | ⚪ old event | Output: 'old event (auto-skipped)'
[010/193] TS: 10.1118    | ⚪ old event | Output: 'old event (auto-skipped)'
[011/193] TS: 10.112236  | ⚪ old event | Output: 'o

In [ ]:
import xml.etree.ElementTree as ET
import io
from google.colab import files

def restructure_xml_to_events(input_xml_content, output_xml_path, new_event_timestamps):
    # Parse directly from the in-memory string from your previous block
    tree = ET.parse(io.StringIO(input_xml_content))
    root = tree.getroot()

    # Create the new structured root
    new_root = ET.Element(root.tag, root.attrib)
    trigger_timestamps = set(str(ts).strip() for ts in new_event_timestamps)

    current_event = None

    # Iterate and group into <event> tags
    for child in root:
        ts = child.get("timestamp")

        if current_event is None or ts in trigger_timestamps:
            current_event = ET.SubElement(new_root, "event")

        current_event.append(child)

    # Save the file
    new_tree = ET.ElementTree(new_root)
    if hasattr(ET, "indent"):
        ET.indent(new_tree, space="  ", level=0)

    new_tree.write(output_xml_path, encoding="utf-8", xml_declaration=True)
    print(f"✅ Successfully created structured XML: {output_xml_path}")


# ==========================================
# REUSE VARIABLES FROM PREVIOUS BLOCK
# ==========================================

# Dynamically generate the output filename using the 'filename' variable already in memory
if filename.endswith(".xml"):
    output_filename = filename.replace(".xml", "_parsed.xml")
else:
    output_filename = filename + "_parsed.xml"

# Run the restructuring function using the 'xml_content' and 'collected_new_events' already in memory
restructure_xml_to_events(xml_content, output_filename, collected_new_events)

# Automatically trigger the browser to download the parsed file
files.download(output_filename)

✅ Successfully created structured XML: first_julia_recording.asciinema_parsed.xml


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

*Model 1 Logic starts*

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.environ["HF_HOME"] = "/content/drive/MyDrive/huggingface_cache"

!pip install -q vllm lxml

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 143.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [ ]:
%%writefile annotator.py
import argparse, json, os, re
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from lxml import etree
from vllm import LLM, SamplingParams

# --- A100 Configuration ---
MODEL_ID = os.getenv("MODEL_ID", "openai/gpt-oss-20b")
GPU_UTIL = float(os.getenv("GPU_UTIL", "0.70"))
MAX_MODEL_LEN = int(os.getenv("MAX_MODEL_LEN", "16384"))
DTYPE = os.getenv("DTYPE", "bfloat16")
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "2000")) # Doubled so it doesn't get cut off while thinking
SUMMARY_WORD_LIMIT = int(os.getenv("SUMMARY_WORD_LIMIT", "50"))
TP_SIZE = int(os.getenv("VLLM_TP", "1"))
K_TARGET = 1
N_NEIGH = 20

_GLOBAL_LLM: Optional[LLM] = None

# --- Prompt Engine & Few Shots ---
FEWSHOTS_BLOCK = """
EXAMPLES (for reference only)

EXAMPLE A — Starting a backup subtask (depth=-1)
neighbor_tail:
  - id=0 depth=0  summary="List project directory contents"
  - id=1 depth=0  summary="Inspect size of source and data folders"
currDepth: 0
input xml:
<event>
  <user_input>t</user_input><system_output>t</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>r</user_input><system_output>r</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>-</user_input><system_output>-</system_output>
  <user_input>c</user_input><system_output>c</system_output>
  <user_input>z</user_input><system_output>z</system_output>
  <user_input>f</user_input><system_output>f</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>b</user_input><system_output>b</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>c</user_input><system_output>c</system_output>
  <user_input>k</user_input><system_output>k</system_output>
  <user_input>u</user_input><system_output>u</system_output>
  <user_input>p</user_input><system_output>p</system_output>
  <user_input>.</user_input><system_output>.</system_output>
  <user_input>t</user_input><system_output>t</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>r</user_input><system_output>r</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>s</user_input><system_output>s</system_output>
  <user_input>r</user_input><system_output>r</system_output>
  <user_input>c</user_input><system_output>c</system_output>
  <system_output>Creating backup.tar...</system_output>
</event>
output:
{"annotation": "Create compressed backup archive of source data", "depth": -1}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE B — Continuing backup subtask (depth=0)
neighbor_tail:
  - id=2 depth=-1 summary="Create compressed backup archive of source data"
currDepth: -1
input xml:
<event>
  <user_input>l</user_input><system_output>l</system_output>
  <user_input>s</user_input><system_output>s</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>-</user_input><system_output>-</system_output>
  <user_input>l</user_input><system_output>l</system_output>
  <user_input>h</user_input><system_output>h</system_output>
  <user_input> </user_input><system_output> </system_output>
  <user_input>b</user_input><system_output>b</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>c</user_input><system_output>c</system_output>
  <user_input>k</user_input><system_output>k</system_output>
  <user_input>u</user_input><system_output>u</system_output>
  <user_input>p</user_input><system_output>p</system_output>
  <user_input>.</user_input><system_output>.</system_output>
  <user_input>t</user_input><system_output>t</system_output>
  <user_input>a</user_input><system_output>a</system_output>
  <user_input>r</user_input><system_output>r</system_output>
  <system_output>-rw-r--r-- 1 user staff 42M backup.tar</system_output>
</event>
output:
{"annotation": "Verify backup archive and check file size", "depth": 0}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE C — Finishing backup subtask (depth=+1)
neighbor_tail:
  - id=2 depth=-1 summary="Create compressed backup archive of source data"
  - id=3 depth=0  summary="Verify backup archive and check file size"
currDepth: -1
input xml:
<event>
  <user_input>m</user_input><user_input>v</user_input><user_input> </user_input>
  <user_input>b</user_input><user_input>a</user_input><user_input>c</user_input>
  <system_output>Moved to archive/</system_output>
</event>
output:
{"annotation": "Move backup to archive folder and complete backup task", "depth": 1}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE D — Starting test/debug subtask (depth=-1)
currDepth: 0
input xml:
<event>
  <user_input>p</user_input><user_input>y</user_input><user_input>t</user_input><user_input>e</user_input><user_input>s</user_input><user_input>t</user_input>
  <system_output>===== test session starts =====</system_output>
</event>
output:
{"annotation": "Start pytest test run for project", "depth": -1}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE E — Nested editor within environment setup (depth=-1)
currDepth: -1
input xml:
<event>
  <user_input>v</user_input><user_input>i</user_input><user_input>m</user_input>
  <system_output>Opening vim...</system_output>
</event>
output:
{"annotation": "Open config file in vim during environment setup", "depth": -1}

═══════════════════════════════════════════════════════════════════════════════

EXAMPLE F — Exit editor, stay in parent task (depth=+1)
currDepth: -2
input xml:
<event>
  <user_input>:</user_input><user_input>w</user_input><user_input>q</user_input>
  <system_output>(venv) $</system_output>
</event>
output:
{"annotation": "Save config changes and exit vim", "depth": 1}
"""

@dataclass
class Event:
    idx: int
    xml: str
    depth_xml: Optional[int] = None
    summary_xml: Optional[str] = None

events: List[Event] = []
pred: Dict[int, Dict[str, object]] = {}

def load_events(xml_path: str) -> List[Event]:
    tree = etree.parse(xml_path)
    root = tree.getroot()
    out = []
    for i, ev_el in enumerate(root.xpath("//event")):
        xml_str = etree.tostring(ev_el, encoding="unicode")
        out.append(Event(idx=i, xml=xml_str))
    return out

def compute_curr_depth_upto(idx: int) -> int:
    curr = 0
    for i in range(idx):
        dep = events[i].depth_xml
        if dep is None: continue
        if dep == -1: curr -= 1
        elif dep > 0: curr += dep
    return curr

def make_flush_package(upto_idx: int, K: int = 1, N: int = 10) -> Dict:
    target_idxs = list(range(max(0, upto_idx - K + 1), upto_idx + 1))
    start_neigh = max(0, target_idxs[0] - N)
    neighbor_idxs = list(range(start_neigh, target_idxs[0]))

    def get_sum(i): return events[i].summary_xml if (0 <= i < len(events) and events[i].summary_xml) else "???"
    def get_dep(i): return events[i].depth_xml if (0 <= i < len(events) and events[i].depth_xml is not None) else 999

    neighbor_info = [f"- id={i} depth={get_dep(i)}  summary={get_sum(i)}" for i in neighbor_idxs]
    target_events = [events[i].xml for i in target_idxs if 0 <= i < len(events)]

    return {
        "target_idxs": target_idxs,
        "neighbor_info": neighbor_info,
        "target_events": target_events,
        "currDepth": compute_curr_depth_upto(target_idxs[0]),
    }

def build_instruction(pkg: dict) -> str:
    # --- NEW: AGGRESSIVE MASSIVE EVENT TRUNCATION ---
    MAX_CHARS = 3000 # Roughly 750 tokens
    truncated_targets = []

    for i, xml in zip(pkg["target_idxs"], pkg["target_events"]):
        if len(xml) > MAX_CHARS:
            # Keep the first 1000 chars (the command) and last 2000 chars (the final result)
            snipped_xml = (
                xml[:1000] +
                f"\n\n... [ ✂️ SYSTEM OUTPUT TRUNCATED - HIDDEN {len(xml) - MAX_CHARS} CHARACTERS ] ...\n\n" +
                xml[-2000:]
            )
            truncated_targets.append(f'<target id="{i}">\n{snipped_xml}\n</target>')
        else:
            truncated_targets.append(f'<target id="{i}">\n{xml}\n</target>')

    targets_xml = "\n".join(truncated_targets)

    return f"""<task>
You are an expert terminal session annotator. Identify goals/subgoals and generate concise action summaries.
</task>

<think_first>
- Keep reasoning CONCISE and FOCUSED
- In <think>...</think>: analyze the command, check depth logic, then conclude
- Aim for 2-3 sentences of reasoning maximum
- Skip obvious observations
- Use neighbors ONLY for continuity; do not invent context.
</think_first>

<rules>
- the user's keystrokes appear separately; combine them to form the full command before interpreting it
- depth is an integer (≥ -1); -1 for subevent (new task started), 0 for same level (still doing the same task), >0 to exit levels (ended one or multiple tasks)
- maintain stack invariant: currDepth ≤ 0; if depth == -1 then currDepth -= 1; if depth > 0 then currDepth += depth
- write action-oriented summaries; avoid "user", "they", "typed", "inputs", "enters a command
- depth is relative to the previous events and nothing else"
</rules>

<output_format>
You MUST wrap your reasoning in <think>...</think> tags.
After the closing </think> tag, you MUST output EXACTLY ONE valid JSON object on a new line. Do not output anything after the JSON.
{{"annotation": "<action summary ≤{SUMMARY_WORD_LIMIT} words>", "depth": <integer ≥ -1>}}
</output_format>

DEPTH SEMANTICS:
- depth = -1: STARTING a new subtask (entering deeper level)
- depth = 0:  CONTINUING at same level (ongoing work)
- depth = +1: FINISHING a subtask (returning to parent level)

<examples>
{FEWSHOTS_BLOCK}
</examples>

<inputs>
  <curr_depth>{pkg.get("currDepth", 0)}</curr_depth>
  <target_events>\n{targets_xml}\n  </target_events>
</inputs>"""

def load_model() -> LLM:
    global _GLOBAL_LLM
    if _GLOBAL_LLM is not None: return _GLOBAL_LLM
    _GLOBAL_LLM = LLM(
        model=MODEL_ID, tensor_parallel_size=TP_SIZE, gpu_memory_utilization=GPU_UTIL,
        max_model_len=MAX_MODEL_LEN, trust_remote_code=True, dtype=DTYPE
    )
    return _GLOBAL_LLM

def generate_with_thinking(llm: LLM, messages: List[Dict[str, str]]):
    tokenizer = llm.get_tokenizer()
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    sampling_params = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS, repetition_penalty=1.2)
    outputs = llm.generate([prompt], sampling_params)
    text = outputs[0].outputs[0].text.strip()
    return text, text, 0, 0

def parse_depth_summary_pairs(text: str) -> List[Tuple[int, str]]:
    dec = json.JSONDecoder()
    out = []

    # NEW PARSING LOGIC: Strip away everything before and inside the <think> tags
    if "</think>" in text:
        text = text.split("</think>")[-1].strip()

    i = 0
    while True:
        start = text.find("{", i)
        if start == -1: break
        try:
            obj, end = dec.raw_decode(text, start)
            if isinstance(obj, dict) and "depth" in obj and "annotation" in obj:
                out.append((int(obj["depth"]), str(obj["annotation"])))
            i = end
        except:
            i = start + 1
    return out

def run_flushes(evs: List[Event]) -> Dict:
    global events, pred
    events = evs
    llm = load_model()

    for upto in range(len(events)):
        success = False
        text = ""
        pkg = {}

        # DYNAMIC CONTEXT SHRINKING: Step down the neighbors if the token limit is exceeded
        for attempt_N in [10, 5, 2, 0]:
            pkg = make_flush_package(upto, N=attempt_N)
            instr = build_instruction(pkg)

            try:
                _, text, _, _ = generate_with_thinking(llm, [{"role": "user", "content": instr}])
                success = True
                break  # Inference succeeded, exit the retry loop
            except Exception as e:
                # Catch the context length error and try again with fewer neighbors
                if "context length" in str(e).lower() or "vllmvalidationerror" in str(type(e)).lower():
                    print(f"\n⚠️ EVENT {upto} EXCEEDED 16K TOKENS (N={attempt_N}). Shrinking history and retrying...")
                    continue
                else:
                    raise e  # If it's a completely different error, crash normally

        if not success:
            print(f"\n❌ EVENT {upto} IS TOO LARGE EVEN WITH 0 NEIGHBORS. Falling back to default.")
            text = '{"annotation": "Event payload too large for context window.", "depth": 0}'

        print(f"\n--- RAW AI OUTPUT FOR EVENT {upto} ---")
        print(text)
        print("--------------------------------------\n")

        pairs = parse_depth_summary_pairs(text)
        if pairs:
            depth, ann = pairs[0]
            pred[pkg["target_idxs"][0]] = {"depth": depth, "summary": ann}
            events[pkg["target_idxs"][0]].depth_xml = depth
            events[pkg["target_idxs"][0]].summary_xml = ann
            print(f"✅ SUCCESSFULLY PARSED -> Depth: {depth} | {ann}")
        else:
            print(f"❌ FAILED TO PARSE JSON FROM EVENT {upto}")

    return pred

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("input_xml")
    parser.add_argument("output_path")
    args = parser.parse_args()

    evs = load_events(args.input_xml)
    preds = run_flushes(evs)

    with open(args.output_path, "w") as f:
        for idx, v in preds.items():
            f.write(json.dumps({"idx": idx, **v}) + "\n")
    print(f"\nSaved to {args.output_path}")

if __name__ == "__main__":
    main()

Overwriting annotator.py


In [ ]:
input_filename = output_filename

# Run the script via command line
!python annotator.py {input_filename} output.jsonl

# Trigger browser download of the results
files.download('output.jsonl')

2026-03-16 16:37:42.885900: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773679062.910753   15964 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773679062.918018   15964 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773679062.935424   15964 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773679062.935454   15964 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773679062.935457   15964 computation_placer.cc:177] computation placer alr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
"""
EndToEnd output from the raw xml file, after processing through model 0 and model 1
"""
import json
# Define the output file name used in the previous step
output_file = 'output.jsonl'

print(f"--- FINAL END-TO-END OUTPUT ({output_file}) ---\n")

try:
    with open(output_file, 'r') as f:
        for line in f:
            # Parse each line as JSON to ensure it's valid, then print
            data = json.loads(line)
            print(json.dumps(data, indent=2))
            print("-" * 40)
except FileNotFoundError:
    print(f"Error: {output_file} not found. Make sure the annotator script finished successfully.")

--- FINAL END-TO-END OUTPUT (output.jsonl) ---

{
  "idx": 0,
  "depth": -1,
  "summary": "Initiate SSH session to host \"1\""
}
----------------------------------------
{
  "idx": 1,
  "depth": 0,
  "summary": "Input numeric sequence into active field"
}
----------------------------------------
{
  "idx": 2,
  "depth": 0,
  "summary": "Enter password for demo@10.0.7.138 via SSH"
}
----------------------------------------
{
  "idx": 3,
  "depth": 0,
  "summary": "View system info upon logging into demo account"
}
----------------------------------------
{
  "idx": 4,
  "depth": 0,
  "summary": "Begin executing a privileged command"
}
----------------------------------------
{
  "idx": 5,
  "depth": 0,
  "summary": "Start pip install of required packages"
}
----------------------------------------
{
  "idx": 6,
  "depth": 0,
  "summary": "Begin typing the command or filename \u2018cinema\u2019"
}
----------------------------------------
{
  "idx": 7,
  "depth": 0,
  "summary": "Supply s